<a href="https://colab.research.google.com/github/beecall17/Fuse-AI-Fellowship/blob/main/Fine%20Tuning%20Transformers%20%5BWeek%2014%5D/support_routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Approach 1: Fine-Tuning Encoder-Only Transformers
### bert-base-uncased

In [3]:
# Imports
from datasets import DatasetDict, load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import numpy as np


In [4]:
# STEP 1: Load and Split the Dataset

raw_datasets = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)
display(raw_datasets)

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})

In [5]:
# Inspect first two rows
display(raw_datasets["train"][:2])

{'flags': ['B', 'BQZ'],
 'instruction': ['question about cancelling order {{Order Number}}',
  'i have a question about cancelling oorder {{Order Number}}'],
 'category': ['ORDER', 'ORDER'],
 'intent': ['cancel_order', 'cancel_order'],
 'response': ["I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.",
  "I've been informed that you have a question about canceling order {{Order Number}}. I'm here to assist you! Please go ahead and let me know what specific question you have, and I'll provide you with all the information and guidance you need. Your satisfaction is my top priority."]}

In [7]:
# Convert category column into ClassLabel so it supports stratification
encoded_dataset = raw_datasets["train"].class_encode_column("category")

# First split: 80% train, 20% temp
split_1 = encoded_dataset.train_test_split(
    test_size=0.20, stratify_by_column="category", seed=42
)

# Second split: Split the 20% temp set equally into 10% validation and 10% test
split_2 = split_1["test"].train_test_split(
    test_size=0.50, stratify_by_column="category", seed=42
)

# Bundle into DatasetDict
df = DatasetDict(
    {"train": split_1["train"], "validation": split_2["train"], "test": split_2["test"]}
)

display(df)

Casting to class labels:   0%|          | 0/26872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 21497
    })
    validation: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 2687
    })
    test: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 2688
    })
})

In [8]:
# STEP 2: Tokenization & Label Preparation
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def preprocess_function(example):
  # Tokenize the instruction (the customer message to route)
  # Note: For routing, only the 'instruction' is used.
  result = tokenizer(example["instruction"], truncation=True)

  # Map the encoded integer category to 'label' as required by HF classification models
  result["label"] = example["category"]
  return result

# Map across all splits and remove raw text columns to keep only tensors
tokenized_datasets = df.map(
    preprocess_function,
    batched=True,
    remove_columns=df["train"].column_names,  # Removes old string columns
)

# STEP 3: Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/21497 [00:00<?, ? examples/s]

Map:   0%|          | 0/2687 [00:00<?, ? examples/s]

Map:   0%|          | 0/2688 [00:00<?, ? examples/s]

In [9]:
# Access the first sample in the training split
sample = tokenized_datasets["train"][0]

# See what keys are available (e.g., input_ids, attention_mask, token_type_ids)
print(sample.keys())

# Check the length of the tokenized instruction/response pair
print("Length of input_ids:", len(sample["input_ids"]))

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'label'])
Length of input_ids: 26


In [11]:
# STEP 4: Load Model for Classification

# Extract labels metadata from the dataset split
num_labels = len(df["train"].features["category"].names)
id2label = {
    i: label for i, label in enumerate(df["train"].features["category"].names)
}
label2id = {
    label: i for i, label in enumerate(df["train"].features["category"].names)
}

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=num_labels, id2label=id2label, label2id=label2id
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
